# 00 · Setup

*Copyright 2026 NetApp Instaclustr. Licensed under the Apache License, Version 2.0. See the `LICENSE` file in this folder.*

This is the shared setup for a workshop on **retrieval for RAG** (retrieval-augmented generation). The running example is a **Recipe & Dietary Safety Assistant**: it answers questions such as "does this recipe contain sulfites?", which have a factual answer, so every failure and every fix is easy to check. This is a teaching demo, not dietary-safety advice.

Everything runs **live** against a shared Instaclustr OpenSearch cluster that already holds a small recipe dataset (260 recipes). The notebooks are the query-only version of the [`rag-retrieval` journey](https://github.com/instaclustr/labs/tree/main/rag-retrieval). The journey covers the full build (ingest, containers, tests). These notebooks cover only the searching.

| Notebook | What you will see |
| --- | --- |
| `01_vector_search` | The data, and why plain vector search retrieves *similar-sounding* recipes, not the right one. |
| `02_bm25_ner` | Keyword search plus entity boosting finds both same-named recipes. |
| `03_hybrid_search` | BM25 and vector fused inside OpenSearch, in one query. |
| `04_ner_grounding_rerank` | Hard filters guarantee an exclusion such as "no sulfites". |
| `05_governed_retrieval` | Hot and long-term tiers, escalation, and citation checks. |
| `06_monitoring_and_comparison` | Reading the governance signals back, then all strategies side by side. |

**You do not need to run this notebook yourself.** Each of the other notebooks starts by running it for you. Open this one to change a setting, or to see what the setup does.

**Cost and safety.**
- The cluster account is **read-only**. These notebooks only search. They never create an index, write a document, or change a pipeline.
- Searches are free. Generating an answer calls your LLM provider, which **bills you**. That is controlled by one switch, `RUN_LLM`, which is `False` by default.

## Install and open

Open the notebooks from the folder that holds `requirements.txt`, with a Python 3.10+ environment active:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
python -m spacy download en_core_web_sm
jupyter lab
```

Keep all the notebook files in one folder, because they find `00_setup.ipynb` by its relative path.

## Settings

The next cell holds the two switches and the connection details.

- `USE_SHARED_READONLY_CLUSTER` points the notebooks at the shared workshop cluster with a read-only account. The account and password below are shared with everyone who uses these notebooks, so they are not secrets. **Do not copy this pattern for real credentials.** Set it to `False` to use your own cluster instead, read from `OPENSEARCH_*` variables in your environment or in `.env`.
- `RUN_LLM` turns answer generation on. Leave it `False` to run retrieval only. To turn it on you need an LLM provider in a `.env` file in this folder (see the cell after next).

In [ ]:
import os

# --- Switches ---------------------------------------------------------------
USE_SHARED_READONLY_CLUSTER = True   # False: use your own cluster from OPENSEARCH_* env vars
RUN_LLM = False                      # True: also generate answers (calls your LLM provider and bills you)

# --- Shared read-only workshop cluster (not a secret; see the note above) ----
SHARED_READONLY = {
    "host": "load-balancer.1121d52d3cf245188b08bae9d912a2bf.cnodes.io",
    "port": 9200,
    "user": "query_readonly",
    "password": "bxgWGyDmPm12RiWgXNJV",
    "ssl": True,
}
SHARED_INDEXES = {
    "vector_baseline": "ragretrieval-recipes-vector-baseline",
    "bm25_baseline": "ragretrieval-recipes-bm25-baseline",
    "hybrid": "ragretrieval-recipes-hybrid",
    "bm25": "ragretrieval-recipes-bm25",
    "vector": "ragretrieval-recipes-vector",
    "audit": "ragretrieval-recipes-audit-log",
}

### Your LLM settings (only needed when `RUN_LLM = True`)

The notebook reads your LLM settings from a `.env` file in this folder (or from environment variables). The file is git-ignored; never commit it. Copy `.env.example` to `.env` and fill it in. It needs one of these sets:

```bash
# Amazon Bedrock (the default)
AWS_ACCESS_KEY_ID=...
AWS_SECRET_ACCESS_KEY=...
AWS_REGION=us-east-1
BEDROCK_MODEL_ID=...        # a Claude model you have enabled in Bedrock

# or an OpenAI-compatible API
LLM_PROVIDER=openai
OPENAI_API_KEY=...
OPENAI_MODEL=...
```

Optional: `HF_TOKEN` gives higher download limits for the embedding model, and `EMBEDDING_MODEL` changes it (the indexes were built with `sentence-transformers/all-MiniLM-L6-v2`, so keep the default).

In [ ]:
import json, re, time, warnings
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from opensearchpy import OpenSearch

warnings.filterwarnings("ignore")
load_dotenv(find_dotenv(usecwd=True))   # real environment variables win over .env

if USE_SHARED_READONLY_CLUSTER:
    OS = SHARED_READONLY
    IDX = SHARED_INDEXES
else:
    OS = {
        "host": os.environ["OPENSEARCH_HOST"],
        "port": int(os.getenv("OPENSEARCH_PORT", "9200")),
        "user": os.environ["OPENSEARCH_USER"],
        "password": os.environ["OPENSEARCH_PASSWORD"],
        "ssl": os.getenv("OPENSEARCH_SSL", "true").lower() in ("1", "true", "yes", "on"),
    }
    IDX = {
        "vector_baseline": os.getenv("OPENSEARCH_VECTOR_RECIPES_BASELINE_INDEX", "recipes-vector-baseline"),
        "bm25_baseline": os.getenv("OPENSEARCH_BM25_RECIPES_BASELINE_INDEX", "recipes-bm25-baseline"),
        "hybrid": os.getenv("OPENSEARCH_HYBRID_INDEX", "recipes-hybrid"),
        "bm25": os.getenv("OPENSEARCH_RECIPE_BM25_INDEX", "recipes-bm25"),
        "vector": os.getenv("OPENSEARCH_RECIPE_VECTOR_INDEX", "recipes-vector"),
        "audit": os.getenv("OPENSEARCH_AUDIT_INDEX", "recipes-audit-log"),
    }

client = OpenSearch(
    hosts=[{"host": OS["host"], "port": OS["port"]}],
    http_auth=(OS["user"], OS["password"]),
    use_ssl=OS["ssl"], verify_certs=OS["ssl"], ssl_show_warn=False,
    timeout=60, max_retries=3, retry_on_timeout=True,
)

info = client.info()
print(f"[PASS] cluster reachable: OpenSearch {info['version']['number']}")
for name, index in IDX.items():
    count = client.count(index=index)["count"]
    print(f"[PASS] index {name:<16} {index}  ({count} documents)")

**Expected:** a `[PASS]` line for the cluster and one for each of the six indexes. Five hold 260 documents. The audit-log index holds the governance records written by earlier runs, so its count is small and varies. If the cluster line fails, check your network and VPN, and that the cluster's firewall allows your IP address.

### Load the models

Two small models run **on your machine**:

- **The embedding model** (`all-MiniLM-L6-v2`, about 90 MB, 384 dimensions) turns a question into a vector. The first run downloads it, so it can take a minute.
- **spaCy's English model** (`en_core_web_sm`, about 13 MB) tags the entities in a question, such as a recipe's website name. It plays the role of the journey's separate NER service, but runs in this notebook's own process.

In [ ]:
import spacy
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
model = SentenceTransformer(EMBEDDING_MODEL)
dim = model.get_sentence_embedding_dimension()

mapping = client.indices.get_mapping(index=IDX["vector"])[IDX["vector"]]["mappings"]["properties"]
index_dim = mapping["embedding"]["dimension"]
print(f"[{'PASS' if dim == index_dim else 'FAIL'}] embedding model {EMBEDDING_MODEL}: {dim} dimensions "
      f"(index expects {index_dim})")

nlp = spacy.load("en_core_web_sm")
print(f"[PASS] spaCy {spacy.__version__}: en_core_web_sm loaded")

In [ ]:
import boto3

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "bedrock").strip().lower()
if LLM_PROVIDER == "openai":
    needed = ["OPENAI_API_KEY", "OPENAI_MODEL"]
else:
    needed = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION", "BEDROCK_MODEL_ID"]
missing = [name for name in needed if not os.getenv(name)]

if not RUN_LLM:
    print(f"[SKIP] RUN_LLM is False: answers are off, retrieval still runs (provider would be {LLM_PROVIDER!r})")
elif missing:
    print(f"[FAIL] RUN_LLM is True but these settings are missing: {', '.join(missing)}")
    print("       Add them to .env next to these notebooks. Answers will be skipped until then.")
else:
    print(f"[PASS] LLM provider {LLM_PROVIDER!r}: all required settings are present")

LLM_READY = RUN_LLM and not missing

**Expected:** `[SKIP]` while `RUN_LLM = False`. With `RUN_LLM = True` you want `[PASS]`. The check only looks that the settings exist. It does not call the provider, so it cannot tell you whether your account has access to the model.

### Helper functions

These small helpers are used by every notebook. Read them once. Each one mirrors a function in the journey's `support/common/` code, which is named in the comment.

`VECTOR_SYSTEM` is the instruction given to the model by the baseline strategies. Later notebooks reuse it.

In [ ]:
RECIPE_FIELDS = ["recipe_id", "recipe_name", "source", "text", "cautions_display", "allergens", "cuisine_type", "tier"]
NER_LABELS = {"PERSON", "ORG", "PRODUCT", "GPE", "EVENT", "WORK_OF_ART", "NORP", "LOC", "FAC"}


def embed(text):
    """Question -> normalized vector (journey: support/common/embeddings.py)."""
    vec = model.encode([text], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0]
    return vec.astype(float).tolist()


def extract_entities(text):
    """Entities spaCy finds in the text (journey: scripts/baseline-bm25-rag/src/ner_service.py)."""
    seen, out = set(), []
    for ent in nlp(text).ents:
        name = " ".join(ent.text.strip().lower().split())
        if ent.label_ in NER_LABELS and len(name) >= 3 and name not in seen:
            seen.add(name)
            out.append(name)
    return out


def normalize_hits(response):
    """Flatten OpenSearch hits into plain dicts (journey: support/common/opensearch_client.py)."""
    return [
        {"score": float(h.get("_score") or 0.0), "id": h.get("_id"), **h.get("_source", {})}
        for h in response["hits"]["hits"]
    ]


def search(index, body, **params):
    return client.search(index=index, body=body, request_timeout=30, params=params or None)


def cautions_of(hit):
    """Allergen cautions of a hit. The hybrid index calls the field `allergens` (lowercase)."""
    return hit.get("cautions_display") or [a.capitalize() for a in hit.get("allergens") or []]


def show_hits(hits, limit=10):
    for rank, h in enumerate(hits[:limit], start=1):
        cautions = ", ".join(cautions_of(h)) or "none"
        print(f"  {rank:>2}. {h.get('recipe_name', '?')[:48]:<48} | {h.get('source', '?'):<28} "
              f"| tier={h.get('tier', '-'):<3} | cautions: {cautions[:40]:<40} | {h['score']:.3f}")


def has_source(hits, domain):
    """True if any hit comes from the given website."""
    return any(domain in (h.get("source") or "").lower() for h in hits)


def llm_generate(prompt, system=None):
    """One completion from Bedrock or an OpenAI-compatible API (journey: support/common/llm_client.py)."""
    if LLM_PROVIDER == "openai":
        from openai import OpenAI
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        with OpenAI(base_url=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"),
                    api_key=os.environ["OPENAI_API_KEY"], timeout=60.0, max_retries=0) as oa:
            reply = oa.chat.completions.create(model=os.environ["OPENAI_MODEL"], messages=messages,
                                               max_completion_tokens=1024, temperature=0.2)
        return (reply.choices[0].message.content or "").strip()
    body = {"anthropic_version": "bedrock-2023-05-31", "max_tokens": 1024, "temperature": 0.2,
            "messages": [{"role": "user", "content": prompt}]}
    if system:
        body["system"] = system
    bedrock = boto3.client("bedrock-runtime", region_name=os.environ["AWS_REGION"])
    reply = json.loads(bedrock.invoke_model(modelId=os.environ["BEDROCK_MODEL_ID"], body=json.dumps(body))["body"].read())
    return "".join(p.get("text", "") for p in reply.get("content", []) if p.get("type") == "text").strip()


def answer(prompt, system=None):
    """Generate an answer, or say why not. Never raises, so one failed call does not stop the notebook."""
    if not LLM_READY:
        return "(answer skipped: set RUN_LLM = True and add your LLM settings to .env)"
    try:
        return llm_generate(prompt, system)
    except Exception as exc:  # noqa: BLE001
        return f"(LLM call failed: {type(exc).__name__}: {str(exc)[:200]})"


def build_context(hits, cautions=False, max_chars=900):
    parts = []
    for i, h in enumerate(hits, start=1):
        extra = f" | cautions: {', '.join(cautions_of(h)) or 'none listed'}" if cautions else ""
        parts.append(f"[RECIPE {i} | {h.get('recipe_name', 'unknown')} | source: {h.get('source', 'unknown')}{extra}]\n"
                     f"{(h.get('text') or '')[:max_chars]}")
    return "\n\n".join(parts)


VECTOR_SYSTEM = ("You are a fact-focused recipe assistant. If the answer is not grounded in the snippets, respond with "
                 "'I don't know.' Be precise about which recipe and source you are answering about -- do not blend "
                 "facts from different recipes together.")


Q_GRATEFUL = "Does the thegratefulgirlcooks.com Fish and Chips recipe contain sulfites?"
Q_WOMENS = "Does the womensweeklyfood.com.au Fish and Chips recipe contain sulfites?"
print("Helpers ready.")

## You are set up

If you saw `[PASS]` for the cluster, the six indexes, the embedding model and spaCy, you are ready. Open `01_vector_search.ipynb` to start.

To turn on LLM answers later: copy `.env.example` to `.env`, fill in your provider settings, set `RUN_LLM = True` in the settings cell above, **save the file** (Ctrl/Cmd-S), and run the cell again. The other notebooks run the saved copy of this one, so an unsaved change has no effect on them. A full pass through notebooks 01 to 05 makes about ten LLM calls.